In [ ]:
from pathlib import Path

import lightning as L
from IPython.display import Video, display

from training.evaluate import evaluate_airtime, load_checkpoint, record_video
from training.ppo import PPO, PPOData
from training.sim import SKILLS, DuckEnv, hardware

checkpoint_path = Path("../../checkpoints/m05_factory_walk.pt")


In [ ]:
course_walk = SKILLS["course-walk"]
skills = {"full_reward": course_walk, "no_airtime": course_walk.with_reward_weights(air_time=0.0)}
walk = SKILLS["walk"]
hw = hardware.load("2026-11-26-hd1910")
def start_training(skill):
    L.seed_everything(1, verbose=False)
    net, checkpoint = load_checkpoint(checkpoint_path, "cuda")
    net.train()
    env = DuckEnv(skill, hw, num_envs=2048, device="cuda", seed=1)
    env.set_curriculum_step(checkpoint["common_step_counter"])
    env.reset()
    env.randomize_episode_starts()
    return env, net


In [ ]:
runs = {}
for label, skill in skills.items():
    env, net = start_training(skill)
    try:
        data = PPOData(env, net)
        model = PPO(net)
        trainer = L.Trainer(max_epochs=300, reload_dataloaders_every_n_epochs=1,
                            accelerator="gpu", devices=1, logger=False,
                            enable_checkpointing=False, enable_model_summary=False)
        trainer.fit(model, data)
        runs[label] = net, env.unwrapped.common_step_counter
    finally:
        env.close()

In [ ]:
for label, (net, clock) in runs.items():
    path = Path(f"m05_{label}.mp4")
    record_video(net, walk, hw, path, common_step_counter=clock)
    display(Video(str(path), embed=True))

In [ ]:
# 同一条前进指令下，比较左右脚腾空多久（毫秒）、抬得多高（毫米）。
{label: evaluate_airtime(net, hw, clock) for label, (net, clock) in runs.items()}